# Laboratorio 2 — Complejidad y búsqueda de hiperparámetros

**Estudiantes:** Danna Garcia (202320823), Julia Ferreira (202012068)

**Caso AlpesPlanck.** Continuando el Lab 1, se exploran enfoques de modelado más flexibles (regresión polinomial y regularizada) para estimar `temp_max_manana`, evaluando su desempeño y estabilidad mediante validación cruzada, búsqueda sistemática de hiperparámetros e intervalos de confianza por bootstrapping.

## 1. Importación de librerías y carga de datos

Se reutiliza el conjunto de datos crudo del Laboratorio 1 (`Datos Lab 1.csv`). El énfasis de este laboratorio no está en la exploración ni el procesamiento de los datos (ya cubierto en el Lab 1), sino en el modelado, la validación y el análisis del desempeño predictivo.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score, validation_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.utils import resample

RUTA_DATOS = 'data/Datos Lab 1.csv'
OBJETIVO = 'temp_max_manana'

datos_originales = pd.read_csv(RUTA_DATOS)
datos_originales.head()


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.50,1000.87,1.3993,0.910860,0.875000,94.8,1.7650,0.7786,...,-0.3618,-0.0223,183.5308,143.0,2009.0,1.0,invierno,January,S,-2.12
1,2009-01-02,999.6006,997.93,1002.65,1.5039,0.920868,86.600000,96.3,2.7588,1.4195,...,0.4267,0.3689,40.8436,144.0,2009.0,2.0,invierno,JULY,NE,-0.82
2,2009-01-03,998.5486,993.05,1002.49,3.1304,76.458100,48.390000,93.9,15.1796,1.2509,...,-0.6993,-0.5268,216.9916,144.0,2009.0,3.0,invierno,January,SO,-0.63
3,2009-01-04,988.5107,985.12,992.93,2.3223,89.417400,97.946704,NaN,4.4904,1.7204,...,-1.1268,-1.0413,222.7419,144.0,2009.0,4.0,invierno,JANUARY,SO,-1.44
4,2009-01-05,990.4057,NaN,997.54,4.2315,86.260400,74.600000,93.2,5.3922,3.8003,...,2.6275,0.2874,6.2418,NaN,2009.0,5.0,East,January,N,-10.88


## 2. Partición de los datos

Se descartan duplicados y filas sin etiqueta (`temp_max_manana`), tal como se justificó en el Lab 1, y se separa en entrenamiento y prueba con la misma semilla (`random_state=42`, `test_size=0.25`) para mantener comparabilidad entre laboratorios.

In [2]:
datos = datos_originales.drop_duplicates().dropna(subset=[OBJETIVO]).copy()

X = datos.drop(columns=[OBJETIVO, 'fecha'])
y = datos[OBJETIVO]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f'Entrenamiento: {X_train.shape[0]} filas')
print(f'Prueba: {X_test.shape[0]} filas')


Entrenamiento: 1857 filas
Prueba: 620 filas


## 3. Limpieza de datos (recuperada del Lab 1)

Se recuperan tal cual las funciones de limpieza ya justificadas en el Lab 1: `limpiar_valores_numericos` convierte en ausentes los valores físicamente imposibles (rangos definidos en `RANGOS_VALIDOS`), y `limpiar_predictoras` además normaliza las variantes de escritura de las variables categóricas (`estacion_anio`, `mes`, `sector_viento`). Se empaquetan en `limpiador`, un `FunctionTransformer` que se usará como primer paso dentro de cada pipeline de modelado, para que la limpieza se aplique de forma consistente al entrenar y al predecir.

In [3]:
RANGOS_VALIDOS = {
    'presion_media': (900, 1100), 'presion_min': (900, 1100), 'presion_max': (900, 1100),
    'presion_desv': (0, 100), 'humedad_media': (0, 100), 'humedad_min': (0, 100),
    'humedad_max': (0, 100), 'humedad_desv': (0, 100), 'viento_media': (0, 100),
    'viento_min': (0, 100), 'viento_max': (0, 100), 'viento_desv': (0, 100),
    'rafaga_media': (0, 100), 'rafaga_min': (0, 100), 'rafaga_max': (0, 100),
    'rafaga_desv': (0, 100), 'viento_norte': (-100, 100), 'viento_este': (-100, 100),
    'direccion_viento': (0, 360), 'registros_del_dia': (1, 144),
    'anio': (2003, 2015), 'dia_del_anio': (1, 366)
}

def limpiar_valores_numericos(dataframe):
    limpio = dataframe.copy()
    for columna, (minimo, maximo) in RANGOS_VALIDOS.items():
        limpio.loc[~limpio[columna].between(minimo, maximo), columna] = np.nan
    return limpio

MAPA_ESTACION = {
    'invierno': 'invierno', 'invernio': 'invierno', 'winter': 'invierno',
    'primavera': 'primavera', 'primav': 'primavera', 'primaveraa': 'primavera', 'spring': 'primavera',
    'verano': 'verano', 'berano': 'verano', 'verno': 'verano', 'summer': 'verano',
    'otono': 'otono', 'otoño': 'otono', 'autumn': 'otono', 'fall': 'otono'
}
MAPA_MES = {
    'jan': 'enero', 'january': 'enero', 'enero': 'enero',
    'feb': 'febrero', 'february': 'febrero', 'febrero': 'febrero',
    'mar': 'marzo', 'march': 'marzo', 'marzo': 'marzo',
    'apr': 'abril', 'april': 'abril', 'abril': 'abril',
    'may': 'mayo', 'mayo': 'mayo',
    'jun': 'junio', 'june': 'junio', 'junio': 'junio',
    'jul': 'julio', 'july': 'julio', 'julio': 'julio',
    'aug': 'agosto', 'august': 'agosto', 'agosto': 'agosto',
    'sep': 'septiembre', 'sept': 'septiembre', 'september': 'septiembre', 'septiembre': 'septiembre',
    'oct': 'octubre', 'october': 'octubre', 'octubre': 'octubre',
    'nov': 'noviembre', 'november': 'noviembre', 'noviembre': 'noviembre',
    'dec': 'diciembre', 'december': 'diciembre', 'diciembre': 'diciembre'
}
MAPA_SECTOR = {
    'n': 'N', 'north': 'N', 'norte': 'N',
    'ne': 'NE', 'northeast': 'NE', 'noreste': 'NE',
    'e': 'E', 'east': 'E', 'este': 'E',
    'se': 'SE', 'southeast': 'SE', 'sureste': 'SE',
    's': 'S', 'south': 'S', 'sur': 'S',
    'so': 'SO', 'southwest': 'SO', 'suroeste': 'SO',
    'o': 'O', 'west': 'O', 'oeste': 'O',
    'no': 'NO', 'northwest': 'NO', 'noroeste': 'NO'
}

def limpiar_predictoras(dataframe):
    limpio = limpiar_valores_numericos(dataframe)
    limpio['estacion_anio'] = limpio['estacion_anio'].str.strip().str.lower().map(MAPA_ESTACION)
    limpio['mes'] = limpio['mes'].str.strip().str.lower().map(MAPA_MES)
    limpio['sector_viento'] = limpio['sector_viento'].str.strip().str.lower().map(MAPA_SECTOR)
    return limpio

limpiador = FunctionTransformer(limpiar_predictoras, validate=False)

columnas_numericas = X.select_dtypes(include='number').columns.tolist()
columnas_categoricas = X.select_dtypes(include='object').columns.tolist()
